In [ ]:
# Lab type: review
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Hybrid Retrieval: BM25 + Dense Fusion and RRF
# Task: A working hybrid retriever is below. Run it, then answer the
# judgment questions about its design choices — several are defensible,
# at least two are defects you should be able to name and fix.

# Lab: Reviewing a Hybrid Retriever

This retriever runs and produces plausible results. Your job is judgment: for each design choice, decide whether it is sound, and if not, what breaks and for which queries.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

## The retriever under review

In [ ]:
from rank_bm25 import BM25Okapi

# Index-time tokenisation: lowercase, strip punctuation
def tokenize(text):
    return [t.strip('.,:;()$').lower() for t in text.split()]

bm25 = BM25Okapi([tokenize(t) for t in DOC_TEXTS])

def bm25_search(query, k=5):
    # NOTE the tokenisation used here
    scores = bm25.get_scores(query.split())
    order = np.argsort(scores)[::-1][:k]
    return [(DOC_IDS[i], float(scores[i])) for i in order]

def hybrid_weighted(query, k=5, w=0.5):
    # Combine the two retrievers with a weighted score sum
    combined = {}
    for doc_id, s in dense_search(query, k=10):
        combined[doc_id] = w * s
    for doc_id, s in bm25_search(query, k=10):
        combined[doc_id] = combined.get(doc_id, 0.0) + (1 - w) * s
    ranked = sorted(combined, key=combined.get, reverse=True)
    return ranked[:k]

for q in ["does the teams plan include sso", "what does error E4022 mean"]:
    print(q)
    print("  dense :", [d for d, _ in dense_search(q, k=3)])
    print("  bm25  :", [d for d, _ in bm25_search(q, k=3)])
    print("  hybrid:", hybrid_weighted(q, k=3))

**Question 1.** Print the actual score ranges of the two arms for a few queries. What does `w=0.5` actually weight in `hybrid_weighted`, given those ranges? Which arm dominates the sum, and would changing `w` to 0.9 fix it?

<details>
<summary>🔑 Reveal answer — Question 1</summary>

Dense cosine scores live in roughly 0.3–0.8; BM25 scores on this corpus reach several points and are unbounded. The weighted sum is dominated by BM25 almost regardless of `w` — the weights are decorative because the scales are incompatible. Re-weighting cannot fix a comparison that is meaningless by construction; fusing by *rank* (RRF, Question 3) is the correct repair.

</details>

**Question 2.** Compare the tokenisation at index time (`tokenize`) with the tokenisation at query time inside `bm25_search`. Construct a query where the mismatch matters and demonstrate it.

<details>
<summary>🔑 Reveal answer — Question 2</summary>

The index lowercases and strips punctuation; queries are split raw. `bm25_search("What does Error E4022 mean?")` queries the terms `['What', 'does', 'Error', 'E4022', 'mean?']` — `Error` (capitalised) and `mean?` (punctuation attached) miss the index vocabulary, and on longer identifier queries the damage compounds. The lexical arm silently underperforms on exactly the identifier-style traffic it exists for. Fix: `bm25.get_scores(tokenize(query))` — index and query must share one tokeniser.

</details>

**Question 3.** Implement RRF fusion over the two arms' rankings (`score = Σ 1/(60 + rank)`) and compare its output with `hybrid_weighted` on the eval set. Which queries change, and why?

In [ ]:
# Work here: implement rrf_fuse(rankings, k=60) and compare
# recall@3 of hybrid_weighted vs your RRF fusion over EVAL_SET.


<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def rrf_fuse(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_rrf(query, k=5):
    dense_ids = [d for d, _ in dense_search(query, k=10)]
    bm25_ids = [d for d, _ in bm25_search(query, k=10)]
    return rrf_fuse([dense_ids, bm25_ids])[:k]

for fn in (hybrid_weighted, hybrid_rrf):
    hits = sum(1 for q, rel in EVAL_SET if set(fn(q, k=3)) & rel)
    print(fn.__name__, hits / len(EVAL_SET))
```

On a 12-document corpus both fusions can reach the same recall@3 — the corpus is too small for top-3 to miss much, which is itself a lesson in why demos hide fusion defects. The difference is visible in the *scores*: print `hybrid_weighted`'s combined dict for a paraphrase query and note the dense contribution is numerically irrelevant next to BM25's scale, then look at a query where the arms disagree (the E4022 query, where the raw-token BM25 arm misranks) and see RRF settle it by consensus instead of by whichever scale is bigger. At production corpus sizes that difference is recall, not just margins.

</details>

**Question 4.** This corpus has 12 documents and the demo works either way. Name the two query populations from the lesson that decide whether hybrid retrieval is worth a second index in production, and say how you would measure whether *this* system needs it.

<details>
<summary>🔑 Reveal answer — Question 4</summary>

Identifier-style queries (error codes, part numbers, function names) — where BM25 uniquely wins — and paraphrase queries — where dense uniquely wins. Measure by running the labelled query set through each arm *separately*: if one arm alone matches the fused recall, skip the second index; if each arm uniquely wins a meaningful slice, hybrid is buying exactly that slice.

</details>

## Summary

1. Weighted sums over incompatible score scales are _______, whatever the weights.
2. Index-time and query-time _______ must match for the lexical arm to work.
3. RRF fuses _______, rewarding documents both retrievers agree on.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **meaningless/decorative** — one scale dominates regardless.
2. **tokenisation** — a silent mismatch starves BM25 of matches.
3. **rank positions** — consensus beats confidence within one arm.

</details>